# Phase 1: PTB-XL Data Loading, Preprocessing, and Inspection

This self-contained notebook implements the complete data pipeline for training 1D CNNs on the **PTB-XL ECG dataset**:
1. **Download & Extract PTB-XL** (100 Hz version) directly from PhysioNet.
2. **Parse `ptbxl_database.csv` & `scp_statements.csv`** and convert raw `scp_codes` to the 5 diagnostic superclasses (`NORM`, `MI`, `STTC`, `CD`, `HYP`).
3. **Load 12-lead signals with `wfdb.rdsamp`**, stack to `(N, 1000, 12)`, and transpose to `(N, 12, 1000)` for PyTorch.
4. **Standardize per lead** using the training set statistics (`strat_fold <= 8`).
5. **Plot 12-lead ECGs** to visually inspect signals and diagnostic labels.
6. **Build PyTorch DataLoaders** with the patient-aware `strat_fold` split.

### Step 1: Install & Import Dependencies

In [ ]:
# Install required packages (uncomment if running in Colab)
!pip install -q wfdb torch pandas numpy scikit-learn matplotlib tqdm

import ast
import os
import sys
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
import wfdb

# 5 Diagnostic superclasses in PTB-XL benchmark
SUPERCLASSES = ["NORM", "MI", "STTC", "CD", "HYP"]

# Standard 12-lead ordering in clinical ECGs
STANDARD_LEAD_NAMES = ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"]

print("Libraries successfully imported!")

### Step 2: Download & Extract PTB-XL from PhysioNet

PhysioNet provides the complete dataset archive (~1.7 GB for both 100 Hz and 500 Hz). We download it and extract the files.

In [ ]:
# Download the official archive (~1.7GB) if not already downloaded
!wget -nc -q --show-progress https://physionet.org/static/published-projects/ptb-xl/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3.zip

# Unzip if directory does not exist
archive_dir = Path("ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3")
if not archive_dir.exists():
    print("Extracting dataset archive...")
    !unzip -q -n ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3.zip
    print("Extraction complete!")
else:
    print("Dataset directory already exists.")

DATA_DIR = archive_dir
print("Data directory:", DATA_DIR.resolve())

### Step 3: Load Metadata & Map SCP Codes to 5 Diagnostic Superclasses

`scp_codes` in `ptbxl_database.csv` contains statements such as `{'NORM': 100.0, 'SR': 0.0}`. We cross-reference `scp_statements.csv` (`diagnostic == 1`) to map diagnostic codes to the 5 superclasses: `NORM`, `MI`, `STTC`, `CD`, `HYP`.

In [ ]:
def load_ptbxl_metadata(data_dir: Path) -> Tuple[pd.DataFrame, pd.DataFrame]:
    # 1. Read metadata
    df = pd.read_csv(data_dir / "ptbxl_database.csv", index_col="ecg_id")
    df.scp_codes = df.scp_codes.apply(ast.literal_eval)

    # 2. Read statement dictionary and filter for diagnostic statements
    scp_df = pd.read_csv(data_dir / "scp_statements.csv", index_col=0)
    diagnostic_statements = scp_df[scp_df.diagnostic == 1]
    code_to_class = diagnostic_statements.diagnostic_class.to_dict()

    # 3. Map individual codes to superclasses
    def map_to_superclasses(scp_dict: Dict[str, float]) -> List[str]:
        classes = set()
        for code in scp_dict.keys():
            if code in code_to_class:
                sc = code_to_class[code]
                if sc in SUPERCLASSES:
                    classes.add(sc)
        return list(classes)

    df["diagnostic_superclass"] = df.scp_codes.apply(map_to_superclasses)

    # 4. Multi-hot binary columns
    for sc in SUPERCLASSES:
        df[sc] = df["diagnostic_superclass"].apply(lambda classes: int(sc in classes))

    return df, scp_df

df, scp_df = load_ptbxl_metadata(DATA_DIR)

print(f"Total records loaded: {len(df)}")
print("\nClass distribution across 5 superclasses:")
for sc in SUPERCLASSES:
    count = df[sc].sum()
    pct = (count / len(df)) * 100
    print(f"  {sc:5s}: {count:5d} ({pct:.1f}%)")

df[['patient_id', 'age', 'sex', 'scp_codes', 'strat_fold'] + SUPERCLASSES].head()

### Step 4: Load Raw 12-Lead Signals with `wfdb.rdsamp`

We load the 100 Hz signals (`filename_lr`). Raw shape is `(N, 1000, 12)`. We transpose this to PyTorch standard `(N, 12, 1000)`: `(N, channels, time_steps)`.

In [ ]:
def load_raw_signals(df: pd.DataFrame, data_dir: Path, sampling_rate: int = 100) -> np.ndarray:
    file_col = "filename_lr" if sampling_rate == 100 else "filename_hr"
    signals = []
    
    for rel_path in tqdm(df[file_col], desc="Loading ECG signals"):
        full_path = str(data_dir / rel_path)
        record_data, _ = wfdb.rdsamp(full_path)
        signals.append(record_data)

    signals = np.array(signals, dtype=np.float32)  # shape: (N, 1000, 12)
    signals = np.transpose(signals, (0, 2, 1))      # shape: (N, 12, 1000)
    return signals

X = load_raw_signals(df, DATA_DIR, sampling_rate=100)
y = df[SUPERCLASSES].values.astype(np.float32)

print(f"Signals shape: {X.shape} -> (N={X.shape[0]}, Leads={X.shape[1]}, TimeSteps={X.shape[2]})")
print(f"Labels shape:  {y.shape} -> (N={y.shape[0]}, Classes={y.shape[1]})")

### Step 5: Standardise Per Lead Using Training Set Statistics

To avoid data leakage, we compute the mean and standard deviation for each of the 12 leads **strictly from the training folds** (`strat_fold <= 8`).

In [ ]:
def standardize_per_lead(X: np.ndarray, train_mask: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    X_train = X[train_mask]
    # Compute mean and std per lead across samples and time: axis (0, 2)
    lead_mean = np.mean(X_train, axis=(0, 2), keepdims=True)  # (1, 12, 1)
    lead_std = np.std(X_train, axis=(0, 2), keepdims=True)    # (1, 12, 1)
    lead_std = np.where(lead_std == 0, 1.0, lead_std)

    X_norm = (X - lead_mean) / lead_std
    return X_norm.astype(np.float32), lead_mean, lead_std

train_mask = (df.strat_fold <= 8).values
X_norm, lead_mean, lead_std = standardize_per_lead(X, train_mask)

print("Training Set Lead Means:\n", np.round(lead_mean.squeeze(), 4))
print("\nTraining Set Lead Standard Deviations:\n", np.round(lead_std.squeeze(), 4))
print(f"\nNormalized min: {X_norm.min():.2f}, max: {X_norm.max():.2f}")

### Step 6: 12-Lead ECG Visual Sanity Checks

Let's visualize sample records: a Normal ECG (`NORM`) and a Myocardial Infarction (`MI`) in standard clinical 12-lead layout.

In [ ]:
def plot_12lead_ecg(
    signal: np.ndarray,
    title: str = "12-Lead ECG",
    lead_names: List[str] = STANDARD_LEAD_NAMES,
    sampling_rate: int = 100,
    save_path: Optional[str] = None,
) -> plt.Figure:
    if signal.shape[0] != 12 and signal.shape[1] == 12:
        signal = signal.T

    num_samples = signal.shape[1]
    time_axis = np.arange(num_samples) / sampling_rate

    fig, axes = plt.subplots(6, 2, figsize=(14, 10), sharex=True)
    axes = axes.flatten()

    for i in range(12):
        axes[i].plot(time_axis, signal[i], color="#1f77b4", linewidth=1.1)
        axes[i].set_title(lead_names[i], loc="left", fontsize=11, fontweight="bold", pad=2)
        axes[i].grid(True, linestyle="--", alpha=0.5)
        axes[i].set_ylabel("mV", fontsize=9)

    axes[-2].set_xlabel("Time (seconds)", fontsize=10)
    axes[-1].set_xlabel("Time (seconds)", fontsize=10)
    fig.suptitle(title, fontsize=14, fontweight="bold", y=0.995)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=200, bbox_inches="tight")

    return fig

# 1. Find a pure Normal ECG
norm_idx = np.where((df['NORM'] == 1) & (df[['MI', 'STTC', 'CD', 'HYP']].sum(axis=1) == 0))[0][0]
fig_norm = plot_12lead_ecg(
    X[norm_idx],
    title=f"Sample ECG #{df.index[norm_idx]} - Diagnostic: Normal (NORM)",
    save_path="sample_ecg_norm.png"
)
plt.show()

# 2. Find a Myocardial Infarction ECG
mi_idx = np.where(df['MI'] == 1)[0][0]
active_labels = [sc for sc in SUPERCLASSES if df[sc].iloc[mi_idx] == 1]
fig_mi = plot_12lead_ecg(
    X[mi_idx],
    title=f"Sample ECG #{df.index[mi_idx]} - Diagnostic: {', '.join(active_labels)}",
    save_path="sample_ecg_mi.png"
)
plt.show()

### Step 7: PyTorch Dataset & DataLoaders Verification

We wrap the normalized arrays in PyTorch `Dataset` and `DataLoader` instances:
- **Train**: Folds 1–8
- **Val**: Fold 9
- **Test**: Fold 10

In [ ]:
class PTBXLDataset(Dataset):
    def __init__(self, signals: np.ndarray, labels: np.ndarray):
        self.signals = torch.from_numpy(signals).float()
        self.labels = torch.from_numpy(labels).float()

    def __len__(self) -> int:
        return len(self.signals)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        return self.signals[idx], self.labels[idx]

train_mask = (df.strat_fold <= 8).values
val_mask = (df.strat_fold == 9).values
test_mask = (df.strat_fold == 10).values

train_ds = PTBXLDataset(X_norm[train_mask], y[train_mask])
val_ds = PTBXLDataset(X_norm[val_mask], y[val_mask])
test_ds = PTBXLDataset(X_norm[test_mask], y[test_mask])

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

print(f"Train samples: {len(train_ds)} ({len(train_loader)} batches)")
print(f"Val samples:   {len(val_ds)} ({len(val_loader)} batches)")
print(f"Test samples:  {len(test_ds)} ({len(test_loader)} batches)")

# Check one batch
batch_x, batch_y = next(iter(train_loader))
print("\n--- Single Batch Sanity Check ---")
print("Batch X shape:", batch_x.shape, "-> (Batch Size, Leads, Time Steps)")
print("Batch y shape:", batch_y.shape, "-> (Batch Size, 5 Classes)")
print("First sample multi-hot label vector:", batch_y[0].numpy())